In [ ]:
!pip install xgboost mlflow scikit-learn pandas

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import mlflow
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)
records = []
highways = ['NH-44', 'NH-48', 'NH-16', 'NH-275', 'NH-65']

for _ in range(500):
    highway = np.random.choice(highways)
    rainfall = np.random.choice([
        np.random.uniform(0, 5),
        np.random.uniform(5, 30),
        np.random.uniform(30, 80),
        np.random.uniform(80, 150)
    ], p=[0.4, 0.3, 0.2, 0.1])

    congestion = np.random.uniform(0, 1)
    news_risk = np.random.randint(0, 4)
    temp = np.random.uniform(18, 42)
    humidity = np.random.uniform(30, 95)
    wind = np.random.uniform(0, 60)
    month = np.random.randint(1, 13)

    disruption = 0
    if rainfall > 80:
        disruption = 1
    elif rainfall > 40 and humidity > 75:
        disruption = 1
    elif news_risk >= 2:
        disruption = 1
    elif congestion > 0.7:
        disruption = 1
    elif month in [6, 7, 8, 9] and rainfall > 20:
        disruption = 1

    records.append({
        'highway_id': highways.index(highway),
        'rainfall_mm': round(rainfall, 2),
        'temp_c': round(temp, 2),
        'humidity': round(humidity, 2),
        'wind_kph': round(wind, 2),
        'congestion_level': round(congestion, 2),
        'news_risk_count': news_risk,
        'month': month,
        'disruption': disruption
    })

df = pd.DataFrame(records)
print(df.shape)
print(df['disruption'].value_counts())

X = df.drop('disruption', axis=1)
y = df['disruption']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("freight-risk-prediction")

with mlflow.start_run():
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(classification_report(y_test, y_pred))

with open('freight_risk_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved.")

In [ ]:
from google.colab import files
files.download('freight_risk_model.pkl')